In [ ]:
#Only for 3 categories: Account Issues, UX issues, UX positive as categories as dominant, but vague by themselves.

In [ ]:
!pip install anthropic -q

import anthropic
import pandas as pd
import json
import time
from google.colab import files, userdata

print('Upload your classified CSV file')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_full  = pd.read_csv("bumble_classified_v2.csv")
print(f'Loaded {len(df_full)} rows from {filename}')
print(f'\nCategory distribution:')
print(df_full['category_1'].value_counts().head(10).to_string())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 12.0 MB/s eta 0:00:00
Upload your classified CSV file


Saving bumble_classified_v2.csv to bumble_classified_v2.csv
Loaded 47202 rows from bumble_classified_v2.csv

Category distribution:
category_1
Forced subscription              6506
Uncategorised                    6025
Account issues                   4833
No matches                       2938
Positive emotional experience    2648
UX issues                        2261
Monetisation manipulation        2216
Fake profiles & bots             2077
Poor value for money             1813
Bad quality matches              1732


/tmp/ipykernel_950/3522226057.py:12: DtypeWarning: Columns (2,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full  = pd.read_csv("bumble_classified_v2.csv")


In [ ]:
# ---------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------
MODEL       = 'claude-haiku-4-5-20251001'
OUTPUT_FILE = 'bumble_subcategorised.csv'

client = anthropic.Anthropic(api_key=userdata.get('bumble_API'))
print(f'Client ready. Model: {MODEL}')

# Filter to only the three categories needing subcategories
TARGET_CATEGORIES = ['Account issues', 'UX issues', 'UX positive']
df_target = df_full[df_full['category_1'].isin(TARGET_CATEGORIES)].copy().reset_index(drop=True)
print(f'\nRows to subcategorise: {len(df_target)}')
print(df_target['category_1'].value_counts().to_string())

Client ready. Model: claude-haiku-4-5-20251001

Rows to subcategorise: 7602
category_1
Account issues    4833
UX issues         2261
UX positive        508


In [ ]:
# ---------------------------------------------------------------
# SYSTEM PROMPTS — one per category
# ---------------------------------------------------------------

PROMPTS = {

'Account issues': """You are subcategorising app store reviews and Reddit comments about Bumble that have already been classified as "Account issues".

Choose the single best subcategory from this list:

- "Unjustified ban / suspension" = account banned or suspended with no clear reason, no warning, no explanation. User feels it was unfair.
- "Login & verification failure" = cannot log in, verification code not received, stuck in login loop, phone/Facebook verification broken.
- "Age verification problems" = user is 18+ but app incorrectly flags them as underage. Specific age detection bug.
- "Banned after paying" = user purchased premium subscription and was then immediately or shortly after banned. Combines trust and monetisation failure.
- "Account access / loading failure" = account inaccessible, app won't load after reinstall, data lost, profile disappeared. Technical access failure not related to bans.
- "Poor support" = submitted support requests, got no response, no human support available, automated replies only.

Respond ONLY with JSON: {"subcategory": "..."}""",

'UX issues': """You are subcategorising app store reviews and Reddit comments about Bumble that have already been classified as "UX issues".

Choose the single best subcategory from this list:

- "Performance & loading" = app is slow, laggy, freezes, crashes, won't load, takes too long. Basic technical performance failure.
- "Missing or removed features" = a feature the user valued has been removed or is no longer available (e.g. speed dating, grid view, question prompts, recommend to friend). Feature regression.
- "Swipe & like limitations" = daily swipe limits, running out of likes, accidental swipes, superswipe placed where swipe button was. Directly linked to monetisation friction.
- "Filter & discovery issues" = distance filter missing or broken, location stuck, can't filter by preference, seeing people too far away, no relevant people in feed.
- "Interface & navigation" = confusing layout, poor design, accidental swipes due to UI placement, notification issues, messaging UI problems, dark mode missing, onboarding unclear, timer frustrations, Opening Move confusion.

Respond ONLY with JSON: {"subcategory": "..."}""",

'UX positive': """You are subcategorising app store reviews and Reddit comments about Bumble that have already been classified as "UX positive".

Choose the single best subcategory from this list:

- "Easy to use / intuitive" = user praises the app for being simple, easy to navigate, user friendly, straightforward.
- "Good interface design" = user specifically praises the visual design, layout, interface, colours, or overall look and feel.
- "Feature appreciation" = user praises a specific feature (undo swipe, free messaging, see sent likes, advanced filters, BFF mode, etc).
- "Better than competitors" = user explicitly says Bumble is better than Tinder, Hinge, or other dating apps in terms of UX.
- "Feature requests" = user likes the app overall but suggests a specific improvement (dark mode, email signup, search radius, weight filter). Positive engagement signal.

Respond ONLY with JSON: {"subcategory": "..."}"""

}

def build_message(row):
    """Build input message using post title if available."""
    text = str(row['text'])[:400]
    if pd.notna(row.get('post_title')) and str(row['post_title']).strip():
        return f"POST TITLE: {str(row['post_title'])[:100]}\n\nREVIEW/COMMENT: {text}"
    return f"REVIEW/COMMENT: {text}"

print('Prompts ready.')

Prompts ready.


In [ ]:
# ---------------------------------------------------------------
# TEST ON 5 ROWS PER CATEGORY
# ---------------------------------------------------------------
print('Testing on 5 rows per category...\n')

for cat in TARGET_CATEGORIES:
    print(f'--- {cat} ---')
    sample = df_target[df_target['category_1'] == cat].sample(5, random_state=42)
    for i, (_, row) in enumerate(sample.iterrows()):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=50,
                system=PROMPTS[cat],
                messages=[{"role": "user", "content": build_message(row)}]
            )
            raw    = response.content[0].text.strip().replace('```json','').replace('```','').strip()
            result = json.loads(raw)
            print(f"  {i+1}. [{result['subcategory']}]")
            print(f"     {str(row['text'])[:80]}")
        except Exception as e:
            print(f"  {i+1}. ERROR: {e}")
    print()

print('Test complete. Check results look correct before running full batch.')

Testing on 5 rows per category...

--- Account issues ---
  1. [Login & verification failure]
     Buggy won't let me log in with my cell phone # and I refuse to log in through Fa
  2. [Login & verification failure]
     No email option
  3. [Account access / loading failure]
     ANSWER TO SUPPORT: I CANT. you have me locked out. you wont let me submit my ema
  4. [Login & verification failure]
     I'm not getting a verification code once I put in my number
  5. [Account access / loading failure]
     Why am I not able to open a bumble account ??

--- UX issues ---
  1. [Filter & discovery issues]
     Could have been better if you guys had an option of ethnicity, like asian, Cauca
  2. [Interface & navigation]
     good app except the time limit to chat to someone, not good for the nervous intr
  3. [Interface & navigation]
     Cant see people who liked me. On "liked you" FIX IT NOW
  4. [Performance & loading]
     Overall the app seems nice but can be quite laggy. Just opening th

In [ ]:
# ---------------------------------------------------------------
# SUBMIT FULL BATCH
# ---------------------------------------------------------------
print(f'Building {len(df_target)} batch requests...')

all_requests = []
for idx, row in df_target.iterrows():
    cat = row['category_1']
    all_requests.append({
        "custom_id": str(idx),
        "params": {
            "model":      MODEL,
            "max_tokens": 50,
            "system":     PROMPTS[cat],
            "messages":   [{"role": "user", "content": build_message(row)}]
        }
    })

# Submit in chunks of 5000
CHUNK_SIZE = 5000
chunks     = [all_requests[i:i+CHUNK_SIZE] for i in range(0, len(all_requests), CHUNK_SIZE)]
print(f'Splitting into {len(chunks)} batch(es) of up to {CHUNK_SIZE} requests...')

BATCH_IDS = []
for i, chunk in enumerate(chunks):
    success  = False
    attempts = 0
    while not success and attempts < 3:
        try:
            print(f'Submitting batch {i+1}/{len(chunks)} ({len(chunk)} requests, attempt {attempts+1})...')
            batch = client.messages.batches.create(requests=chunk)
            BATCH_IDS.append(batch.id)
            print(f'  Success: {batch.id}')
            success = True
            time.sleep(3)
        except Exception as e:
            attempts += 1
            print(f'  Failed: {str(e)[:80]}')
            time.sleep(15)
    if not success:
        print(f'  Giving up on batch {i+1}')

print(f'\nBATCH_IDS: {BATCH_IDS}')
print(f'Submitted: {len(BATCH_IDS)}/{len(chunks)} batches')
print('IMPORTANT: Save batch IDs in Colab secrets as batch_id_subcat_1, batch_id_subcat_2 etc')

Building 7602 batch requests...
Splitting into 2 batch(es) of up to 5000 requests...
Submitting batch 1/2 (5000 requests, attempt 1)...
  Success: msgbatch_01D6MjZvkUiCBB22NqbXmLK3
Submitting batch 2/2 (2602 requests, attempt 1)...
  Success: msgbatch_01DmSGZyCVhxzrZb11bV5m7a

BATCH_IDS: ['msgbatch_01D6MjZvkUiCBB22NqbXmLK3', 'msgbatch_01DmSGZyCVhxzrZb11bV5m7a']
Submitted: 2/2 batches
IMPORTANT: Save batch IDs in Colab secrets as batch_id_subcat_1, batch_id_subcat_2 etc


In [ ]:
# ---------------------------------------------------------------
# CHECK STATUS — re-run until All complete: True
# If session reset:
BATCH_IDS = [userdata.get('batch_id_subcat1'), userdata.get('batch_id_subcat2')]
BATCH_IDS = [x for x in BATCH_IDS if x is not None]
all_done        = True
total_succeeded = 0
total_pending   = 0
total_errored   = 0

for bid in BATCH_IDS:
    status = client.messages.batches.retrieve(bid)
    print(f'{bid}: {status.processing_status} — succeeded: {status.request_counts.succeeded}, pending: {status.request_counts.processing}, errored: {status.request_counts.errored}')
    total_succeeded += status.request_counts.succeeded
    total_pending   += status.request_counts.processing
    total_errored   += status.request_counts.errored
    if status.processing_status != 'ended':
        all_done = False

print(f'\nTotal succeeded: {total_succeeded}')
print(f'Total pending:   {total_pending}')
print(f'All complete:    {all_done}')

msgbatch_01GqCAkPrbEf6LGYcDSJceJd: ended — succeeded: 5000, pending: 0, errored: 0
msgbatch_014Ap83K6sYn6a76PbnHoiyX: ended — succeeded: 2602, pending: 0, errored: 0

Total succeeded: 7602
Total pending:   0
All complete:    True


In [ ]:
# ---------------------------------------------------------------
# RETRIEVE RESULTS
# ---------------------------------------------------------------
print('Retrieving results...')

results = {}
for bid in BATCH_IDS:
    for result in client.messages.batches.results(bid):
        idx = int(result.custom_id)
        if result.result.type == 'succeeded':
            try:
                raw    = result.result.message.content[0].text.strip()
                raw    = raw.replace('```json','').replace('```','').strip()
                parsed = json.loads(raw)
                results[idx] = parsed.get('subcategory', 'Uncategorised')
            except:
                results[idx] = 'Uncategorised'
        else:
            results[idx] = 'Uncategorised'

print(f'Retrieved {len(results)} results')

Retrieving results...
Retrieved 7602 results


In [ ]:
# ---------------------------------------------------------------
# REBUILD df_target EXACTLY AS ORIGINALLY DONE (for index alignment)
# ---------------------------------------------------------------
TARGET_CATEGORIES = ['Account issues', 'UX issues', 'UX positive']
df_target = df_full[df_full['category_1'].isin(TARGET_CATEGORIES)].copy().reset_index(drop=True)
print(f'df_target rows: {len(df_target)}')

# Assign subcategories using positional index (matches results dict keys)
df_target['subcategory'] = [results.get(i, 'Uncategorised') for i in range(len(df_target))]

# ---------------------------------------------------------------
# MERGE BACK USING TEXT+SOURCE AS JOIN KEY (robust to index issues)
# ---------------------------------------------------------------
df_full['subcategory'] = None

merge_map = dict(zip(
    df_target['text'].astype(str) + '|' + df_target['source'].astype(str),
    df_target['subcategory']
))

df_full['_key'] = df_full['text'].astype(str) + '|' + df_full['source'].astype(str)
mask = df_full['category_1'].isin(TARGET_CATEGORIES)
df_full.loc[mask, 'subcategory'] = df_full.loc[mask, '_key'].map(merge_map)
df_full = df_full.drop(columns=['_key'])

print(f'\nSubcategorised: {df_full["subcategory"].notna().sum()} / {mask.sum()} target rows')

# ---------------------------------------------------------------
# TEST — show subcategory counts per category
# ---------------------------------------------------------------
print('\n--- Account issues subcategories ---')
print(df_full[df_full['category_1'] == 'Account issues']['subcategory'].value_counts().to_string())

print('\n--- UX issues subcategories ---')
print(df_full[df_full['category_1'] == 'UX issues']['subcategory'].value_counts().to_string())

print('\n--- UX positive subcategories ---')
print(df_full[df_full['category_1'] == 'UX positive']['subcategory'].value_counts().to_string())

print('\n--- Sanity check: any other categories with subcategory? (should be empty) ---')
other = df_full[(df_full['subcategory'].notna()) & (~df_full['category_1'].isin(TARGET_CATEGORIES))]
print(f'{len(other)} rows')

df_target rows: 7602

Subcategorised: 7602 / 7602 target rows

--- Account issues subcategories ---
subcategory
Login & verification failure        1574
Unjustified ban / suspension        1173
Account access / loading failure    1018
Banned after paying                  461
Age verification problems            351
Poor support                         200
Uncategorised                         56

--- UX issues subcategories ---
subcategory
Interface & navigation         849
Performance & loading          504
Missing or removed features    393
Filter & discovery issues      343
Swipe & like limitations       118
Uncategorised                   51
Onboarding unclear               3

--- UX positive subcategories ---
subcategory
Easy to use / intuitive                     263
Feature requests                            120
Good interface design                        56
Feature appreciation                         53
Better than competitors                      13
Uncategorised           

In [ ]:
df_full.to_csv(OUTPUT_FILE, index=False)
print(f'Saved {len(df_full)} rows to {OUTPUT_FILE}')
print(f'Columns: {list(df_full.columns)}')

Saved 47202 rows to bumble_subcategorised.csv
Columns: ['source', 'original_date', 'comment_date', 'text', 'comment', 'subreddit', 'post_title', 'post_score', 'comment_score', 'rating', 'thumbs_up', 'vader_compound', 'vader_positive', 'vader_negative', 'vader_neutral', 'sentiment_label', 'category_1', 'category_2', 'category_3', 'dimension_1', 'dimension_2', 'classification_confidence', 'Year', 'Fiscal Quarter', 'Financial quarter', 'subcategory']


In [ ]:
files.download(OUTPUT_FILE)
print('Download triggered.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered.


In [ ]:
BATCH_IDS = [userdata.get('batch_id_subcat1'), userdata.get('batch_id_subcat2')]
BATCH_IDS = [x for x in BATCH_IDS if x is not None]
for bid in BATCH_IDS:
    status = client.messages.batches.retrieve(bid)
    print(f'{bid}: {status.processing_status}')
    print(f'Succeeded: {status.request_counts.succeeded}')

msgbatch_01GqCAkPrbEf6LGYcDSJceJd: ended
Succeeded: 5000
msgbatch_014Ap83K6sYn6a76PbnHoiyX: ended
Succeeded: 2602
